# Hito 3 — Pricing Bridge: From LTV to Acquisition and Retention Strategy

## Business framing

Hito 2 gave us LTV predictions per user. A number on its own is not a decision. This notebook converts predicted LTV into three concrete, actionable outputs:

**1. CAC ceiling by segment** — What is the maximum we should spend acquiring a user in each archetype? Overspending on acquisition erodes margin; underspending leaves growth on the table. The LTV predictions give us a principled ceiling for each segment.

**2. Retention discount strategy** — For a dormant user who hasn't bought in 6–12 months, what is the maximum discount worth offering to trigger re-activation? This is directly connected to the core thesis: dormant ≠ churned, so a targeted discount to a high-LTV dormant user is a positive expected-value action — but only up to a point.

**3. Corridor prioritisation** — Which destination corridors should an unlimited-plan provider lean into for growth? Thailand has the best margin rate but lower absolute LTV. Argentina has high revenue but volatile wholesale costs. The ranking combines volume, LTV per user, and margin risk into a single comparable metric.

---

**Connection to the hiring manager's roadmap:**
> Month 2: LTV modelling → Month 3: experimentation over LTV

This notebook is the bridge. Each output here is a hypothesis that can be A/B tested in Month 3: test a higher CAC bid for digital nomads; test a €5 reactivation discount for dormant leisure-repeat users; test corridor-specific acquisition spend allocation.

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.pricing_bridge import (
    compute_cac_ceiling,
    compute_retention_discount,
    rank_corridors,
)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fafafa",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.4,
    "grid.linestyle": "--",
    "font.size": 11,
})

PALETTE = {
    "leisure_once":   "#4C72B0",
    "leisure_repeat": "#DD8452",
    "digital_nomad":  "#55A868",
}
CORRIDOR_PALETTE = {
    "thailand":       "#8172B2",
    "western_europe": "#4C72B0",
    "usa":            "#DD8452",
    "argentina":      "#C44E52",
}

DATA_DIR   = Path("../data/raw")
PROC_DIR   = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("Dependencies loaded ✓")

## 1. Load LTV predictions from Hito 2

In [ ]:
# Load predictions saved by Hito 2
pred_path = PROC_DIR / "ltv_predictions.parquet"
csv_path  = PROC_DIR / "ltv_predictions.csv"

if pred_path.exists():
    ltv = pd.read_parquet(pred_path)
elif csv_path.exists():
    ltv = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(
        "Run notebooks/02_ltv_models.ipynb first to generate ltv_predictions."
    )

print(f"Loaded {len(ltv):,} user predictions")
print(f"Columns: {list(ltv.columns)}")
print(f"\nPredicted LTV summary:")
print(ltv[["predicted_ltv", "label_revenue", "label_margin"]].describe().round(2).to_string())

## 2. CAC ceiling by segment

### Framework

The standard LTV:CAC heuristic is:

> **CAC ceiling = Predicted LTV margin / target LTV:CAC multiple**

We use **margin** (not revenue) as the LTV basis — wholesale cost is stochastic and corridor-dependent, so a €27 plan in Argentina yields far less margin than a €14 plan in Thailand.

We use a **3× multiple** as the conservative baseline. The reasoning: at 3× LTV:CAC, the payback period on a leisure-once user (median ~1 purchase/year) is approximately 12–18 months, which is acceptable given the low churn risk identified in Hito 1. For digital nomads (6 purchases/year), 3× is actually conservative — these users could justify a 5× multiple.

This table is the direct input to a Google Ads or Meta bid strategy: segment users by predicted archetype at acquisition, bid up to the CAC ceiling.

In [ ]:
cac = compute_cac_ceiling(ltv, target_ltv_margin_multiple=3.0)

print("CAC ceiling by archetype (LTV:CAC target = 3×):")
print(cac.to_string(index=False))

In [ ]:
# Also compute for corridor × archetype to show interaction
cac_corridor = compute_cac_ceiling(
    ltv,
    target_ltv_margin_multiple=3.0,
    group_cols=["archetype", "primary_corridor"],
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── (a) CAC ceiling by archetype ─────────────────────────────────────────
ax = axes[0]
archetypes = cac["archetype"].tolist()
x = np.arange(len(archetypes))
w = 0.35

ax.bar(x, cac["cac_ceiling_median"],
       color=[PALETTE[a] for a in archetypes],
       width=0.5, alpha=0.85, edgecolor="white", label="Median CAC ceiling")

# Error bars: IQR
ax.errorbar(
    x, cac["cac_ceiling_median"],
    yerr=[
        cac["cac_ceiling_median"] - cac["cac_ceiling_p25"],
        cac["cac_ceiling_p75"] - cac["cac_ceiling_median"],
    ],
    fmt="none", color="#333", capsize=5, linewidth=1.5,
    label="IQR"
)

ax.set_xticks(x)
ax.set_xticklabels([a.replace("_", "\n").title() for a in archetypes])
ax.set_ylabel("Max justifiable CAC (€)")
ax.set_title("(a) CAC ceiling by archetype\n(3× LTV:CAC target, margin basis)",
             fontweight="bold")
ax.legend(fontsize=9)

# Annotate with LTV
for i, row in cac.iterrows():
    ax.text(i, row["cac_ceiling_median"] + 0.3,
            f"LTV: €{row['median_predicted_ltv']:.0f}",
            ha="center", fontsize=8, color="#444")

# ── (b) CAC ceiling heatmap: archetype × corridor ────────────────────────
ax = axes[1]
pivot = cac_corridor.pivot(
    index="archetype", columns="primary_corridor", values="cac_ceiling_median"
).fillna(0)

sns.heatmap(
    pivot,
    ax=ax,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Median CAC ceiling (€)"},
)
ax.set_title("(b) CAC ceiling (€) by archetype × corridor", fontweight="bold")
ax.set_xlabel("Primary corridor")
ax.set_ylabel("Archetype")
ax.set_xticklabels([t.get_text().replace("_", "\n") for t in ax.get_xticklabels()])

fig.suptitle("Maximum Justifiable Customer Acquisition Cost",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(PROC_DIR / "fig_cac_ceiling.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey insight: digital nomads justify 3-4× higher CAC than leisure-once users.")
print("Argentina corridor suppresses CAC ceiling due to lower margin rate.")

## 3. Retention discount strategy

### Framework

A dormant user (last purchase >180 days ago) represents unrealised LTV. The question is: **how much is it worth spending to reactivate them?**

The break-even condition:

> **EV(offer discount d) = P(reactivate | d) × predicted_margin − d ≥ 0**
> 
> **Max discount = P(reactivate) × predicted_margin**

P(reactivate | d) increases with discount size (logistic function). The curve crosses zero at the max justified discount — beyond that, the expected cost exceeds the expected margin recovery.

Crucially, this framework is testable: offer a range of discount levels (€0, €3, €5, €8) to randomly assigned dormant users and measure the reactivation rate. That's the Month 3 experimentation roadmap.

In [ ]:
user_discounts, discount_curve = compute_retention_discount(
    ltv,
    reactivation_probability=0.25,
    avg_plan_revenue=26.9,
)

dormant = user_discounts[user_discounts["is_dormant"]]

print(f"Dormant users (recency > 180 days, ≥1 prior purchase): {len(dormant):,}")
print(f"\nMax discount by archetype (dormant users only):")
print(
    dormant.groupby("archetype")[["max_discount_eur", "max_discount_pct", "predicted_ltv"]]
    .median()
    .round(2)
    .to_string()
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── (a) EV curves by archetype ───────────────────────────────────────────
ax = axes[0]
for arch, grp in discount_curve.groupby("archetype"):
    ax.plot(
        grp["discount_eur"], grp["expected_value_eur"],
        color=PALETTE.get(arch, "#888"), linewidth=2,
        label=arch.replace("_", " ").title(),
    )
    # Mark break-even
    breakeven = grp[grp["expected_value_eur"] <= 0]["discount_eur"]
    if not breakeven.empty:
        ax.axvline(breakeven.iloc[0], color=PALETTE.get(arch, "#888"),
                   linestyle=":", linewidth=1, alpha=0.6)

ax.axhline(0, color="#999", linewidth=1, linestyle="--")
ax.fill_between(
    discount_curve["discount_eur"].unique(),
    0,
    discount_curve.groupby("discount_eur")["expected_value_eur"].max(),
    alpha=0.05, color="green",
)
ax.set_xlabel("Discount offered (€)")
ax.set_ylabel("Expected value of offer (€)")
ax.set_title("(a) EV of retention discount\nby archetype", fontweight="bold")
ax.legend(fontsize=9)
ax.text(0.05, 0.15, "EV > 0: offer is profitable",
        transform=ax.transAxes, fontsize=8, color="green")
ax.text(0.05, 0.08, "EV < 0: over-discounting",
        transform=ax.transAxes, fontsize=8, color="#C44E52")

# ── (b) Max discount distribution for dormant users ──────────────────────
ax = axes[1]
for arch in ["leisure_once", "leisure_repeat", "digital_nomad"]:
    data = dormant[dormant["archetype"] == arch]["max_discount_eur"]
    ax.hist(
        data,
        bins=20, alpha=0.6,
        label=arch.replace("_", " ").title(),
        color=PALETTE[arch], density=True, edgecolor="white",
    )
ax.set_xlabel("Max justified discount (€)")
ax.set_ylabel("Density")
ax.set_title("(b) Max discount distribution\n(dormant users)", fontweight="bold")
ax.legend(fontsize=9)

# ── (c) Dormant user LTV vs max discount scatter ─────────────────────────
ax = axes[2]
sample = dormant.sample(min(600, len(dormant)), random_state=42)
for arch in ["leisure_once", "leisure_repeat", "digital_nomad"]:
    s = sample[sample["archetype"] == arch]
    ax.scatter(
        s["predicted_ltv"], s["max_discount_eur"],
        alpha=0.4, s=15,
        color=PALETTE[arch],
        label=arch.replace("_", " ").title(),
    )

# Add the decision boundary: "worth targeting" = max_discount > €3
ax.axhline(3, color="#888", linestyle="--", linewidth=1)
ax.text(ax.get_xlim()[1] * 0.05, 3.2,
        "€3 minimum offer threshold", fontsize=8, color="#555")

ax.set_xlabel("Predicted LTV (€)")
ax.set_ylabel("Max justified discount (€)")
ax.set_title("(c) Predicted LTV vs max discount\n(dormant users)", fontweight="bold")
ax.legend(fontsize=9)

fig.suptitle("Retention Discount Strategy — Dormant User Reactivation",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(PROC_DIR / "fig_retention_discount.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Corridor prioritisation

In [ ]:
corridor_ranking = rank_corridors(ltv)

print("Corridor ranking by total predicted margin:")
print(corridor_ranking[[
    "primary_corridor", "n_users", "median_predicted_ltv",
    "median_predicted_margin", "total_predicted_margin",
    "margin_rate", "margin_std", "risk_adjusted_score", "margin_share_pct"
]].to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

corridors_ordered = corridor_ranking["primary_corridor"].tolist()
colors = [CORRIDOR_PALETTE[c] for c in corridors_ordered]

# ── (a) Total predicted margin by corridor ───────────────────────────────
ax = axes[0]
bars = ax.barh(
    corridors_ordered[::-1],
    corridor_ranking["total_predicted_margin"].tolist()[::-1],
    color=colors[::-1], alpha=0.85, edgecolor="white",
)
for bar, v in zip(bars, corridor_ranking["total_predicted_margin"].tolist()[::-1]):
    ax.text(v + 50, bar.get_y() + bar.get_height() / 2,
            f"€{v:,.0f}", va="center", fontsize=8.5)
ax.set_xlabel("Total predicted margin — next 12 months (€)")
ax.set_title("(a) Total margin by corridor", fontweight="bold")

# ── (b) Margin rate vs LTV per user (bubble = n_users) ───────────────────
ax = axes[1]
for _, row in corridor_ranking.iterrows():
    ax.scatter(
        row["margin_rate"] * 100,
        row["median_predicted_ltv"],
        s=row["n_users"] * 0.4,
        color=CORRIDOR_PALETTE[row["primary_corridor"]],
        alpha=0.8, edgecolors="white", linewidth=1,
    )
    ax.annotate(
        row["primary_corridor"].replace("_", "\n"),
        (row["margin_rate"] * 100, row["median_predicted_ltv"]),
        textcoords="offset points", xytext=(6, 3),
        fontsize=8.5,
    )

ax.set_xlabel("Margin rate (%)")
ax.set_ylabel("Median predicted LTV per user (€)")
ax.set_title("(b) Margin rate vs LTV per user\n(bubble size = user volume)",
             fontweight="bold")

# ── (c) Risk-adjusted score ───────────────────────────────────────────────
ax = axes[2]
ras = corridor_ranking[["primary_corridor", "risk_adjusted_score", "margin_std"]]
x = np.arange(len(ras))

bar_colors = [CORRIDOR_PALETTE[c] for c in ras["primary_corridor"]]
bars = ax.bar(x, ras["risk_adjusted_score"],
              color=bar_colors, alpha=0.85, edgecolor="white", width=0.5)
ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", "\n") for c in ras["primary_corridor"]])
ax.set_ylabel("Risk-adjusted score (mean margin / std)")
ax.set_title("(c) Risk-adjusted corridor score\n(higher = better margin per unit risk)",
             fontweight="bold")

# Annotate std
for i, row in ras.iterrows():
    ax.text(list(x)[list(ras.index).index(i)],
            row["risk_adjusted_score"] + 0.002,
            f"σ=€{row['margin_std']:.1f}",
            ha="center", fontsize=8, color="#555")

fig.suptitle("Destination Corridor Prioritisation",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(PROC_DIR / "fig_corridor_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Strategy summary — a decision table

The three analyses above can be combined into a single decision table that a growth or pricing team can act on directly.

In [ ]:
# Build a combined strategy summary
strategy = cac[["archetype", "n_users", "median_predicted_ltv",
                "median_predicted_margin", "cac_ceiling_median"]].copy()

discount_summary = (
    dormant.groupby("archetype")["max_discount_eur"]
    .median()
    .reset_index()
    .rename(columns={"max_discount_eur": "max_retention_discount_eur"})
)

strategy = strategy.merge(discount_summary, on="archetype", how="left")
strategy["max_retention_discount_eur"] = strategy["max_retention_discount_eur"].fillna(0).round(2)

strategy["acquisition_priority"] = pd.cut(
    strategy["cac_ceiling_median"],
    bins=[0, 3, 6, 999],
    labels=["Low", "Medium", "High"]
)

strategy["retention_priority"] = pd.cut(
    strategy["max_retention_discount_eur"],
    bins=[0, 2, 4, 999],
    labels=["Low", "Medium", "High"]
)

print("=" * 80)
print("STRATEGY DECISION TABLE")
print("=" * 80)
print(strategy.to_string(index=False))
print("\nNote: CAC ceiling based on 3× LTV:CAC multiple, margin basis.")
print("Max retention discount = break-even at P(reactivate)=0.25 baseline.")

In [ ]:
# Final summary figure: the full strategy on one chart
fig, ax = plt.subplots(figsize=(10, 5))

archetypes = strategy["archetype"].tolist()
x = np.arange(len(archetypes))
w = 0.25

b1 = ax.bar(x - w, strategy["median_predicted_ltv"],   w,
            label="Predicted LTV (€)", color="#404040", alpha=0.85)
b2 = ax.bar(x,     strategy["cac_ceiling_median"] * 3,  w,
            label="Max CAC × 3 (≈ median margin)",
            color=[PALETTE[a] for a in archetypes], alpha=0.75)
b3 = ax.bar(x + w, strategy["max_retention_discount_eur"] * 5, w,
            label="Max retention discount × 5",
            color=[PALETTE[a] for a in archetypes], alpha=0.40,
            hatch="///", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels([a.replace("_", "\n").title() for a in archetypes])
ax.set_ylabel("Euros (CAC and discount scaled for visibility)")
ax.set_title(
    "LTV-driven strategy by archetype: acquisition ceiling + retention discount",
    fontweight="bold"
)
ax.legend(fontsize=9)

# Annotate actual values
for i, row in strategy.iterrows():
    ax.text(i - w, row["median_predicted_ltv"] + 0.3,
            f"€{row['median_predicted_ltv']:.1f}",
            ha="center", fontsize=8, color="#333")
    ax.text(i, row["cac_ceiling_median"] * 3 + 0.3,
            f"CAC≤€{row['cac_ceiling_median']:.1f}",
            ha="center", fontsize=8, color="#333")
    ax.text(i + w, row["max_retention_discount_eur"] * 5 + 0.3,
            f"Disc≤€{row['max_retention_discount_eur']:.1f}",
            ha="center", fontsize=8, color="#333")

plt.tight_layout()
plt.savefig(PROC_DIR / "fig_strategy_summary.png", dpi=150, bbox_inches="tight")
plt.show()

---

## Key takeaways

1. **CAC ceiling varies 3–4× across archetypes.** Digital nomads justify a significantly higher acquisition spend than leisure-once users — not because they are more valuable per trip, but because their travel frequency means the LTV accrues faster and more reliably. Bid strategies on paid channels should reflect this.

2. **Argentina corridor suppresses both LTV and CAC ceiling.** The expensive wholesale cost (reflected in higher plan prices but lower margin rate) means Argentina-bound users are worth less to acquire and less to retain. Growth budget should tilt toward Thailand and Western Europe.

3. **Retention discounts are only justified for a subset of dormant users.** At a P(reactivate)=0.25 baseline, the break-even discount is €2–5 for leisure segments. This is testable: a simple A/B test offering €0, €3, and €5 discounts to dormant users would directly calibrate the reactivation probability curve used here.

4. **Thailand is the risk-adjusted best corridor.** Despite the lowest absolute LTV, it has the best margin rate (74.8%) and lowest wholesale cost volatility — giving it the highest risk-adjusted score. At scale, this is the safest corridor to grow.

5. **These outputs are Month 3 experiment designs in disguise.** Each number here — CAC ceiling, max discount, corridor ranking — is a hypothesis. The natural next step is to instrument these as treatment arms in a geo-lift test (CAC by segment) and a holdout discount test (dormant reactivation). That is the direct bridge to the a typical Month 3 growth roadmap.

---

**Repository complete.** Three notebooks, three milestones, one coherent argument: travel eSIM LTV requires non-contractual modelling that accounts for annual macro-cycles — and the resulting predictions are directly actionable for acquisition, retention, and corridor strategy.